In [4]:
import os
import pandas as pd
import dateutil
import numpy as np
import shutil

In [10]:
def get_filtered(subdir):
    df = pd.read_csv('output/' + subdir + '/' + subdir + '.csv', parse_dates=['start_time', 'end_time'])
    
    plot_dir = f'output/{subdir}/plots'
    fnames = os.listdir(plot_dir)
    stimes = []
    for fname in fnames:
        toks = fname.split('_')[-2:]
        stime = dateutil.parser.parse(toks[0])
        stimes.append(stime)
        
    mask = df.start_time.apply(lambda s: s in stimes)
    df_filtered = df[mask]

    ordered_fnames = []
    for _, row in df_filtered.iterrows():
        fname = [f for f in fnames if row.start_time.isoformat() in f][0]
        ordered_fnames.append(fname)

    df_filtered['file_name'] = ordered_fnames
    df_filtered.reset_index(inplace=True)
    return df_filtered

In [11]:
df = get_filtered('Sept30_Storm')

In [13]:
df['B'] = np.sqrt(df['Bx']**2 + df['By']**2 + df['Bz']**2)
df['cone_angle'] =  np.rad2deg(np.arccos(df['Bx'] / df['B']))
df['clock_angle'] =  np.rad2deg(np.arctan2(df['By'], df['Bz'])) % 360
df

,index,start_time,end_time,Bx,By,Bz,score,file_name,B,cone_angle,clock_angle
0,0,2025-09-27 01:21:19,2025-09-27 01:22:37,-1.936180,-4.495037,0.769296,0.663395,TracersDispersionEvent_TS2_2025-09-27T01:21:19...,4.954389,113.004350,279.711705
1,5,2025-09-28 01:30:13,2025-09-28 01:31:32,-2.991730,4.040490,-4.695540,0.859231,TracersDispersionEvent_TS2_2025-09-28T01:30:13...,6.879252,115.778408,139.288158
2,20,2025-09-28 16:02:20,2025-09-28 16:03:41,-1.384180,-10.228882,-3.035928,0.573218,TracersDispersionEvent_TS2_2025-09-28T16:02:20...,10.759314,97.391556,253.469133
3,32,2025-09-30 11:28:56,2025-09-30 11:30:19,-8.338225,7.268442,-5.527518,0.688864,TracersDispersionEvent_TS2_2025-09-30T11:28:56...,12.365666,132.400167,127.252329
4,34,2025-10-01 16:29:31,2025-10-01 16:30:54,-6.210000,5.120000,-3.450000,0.664833,TracersDispersionEvent_TS2_2025-10-01T16:29:31...,8.756769,135.167075,123.973199
5,41,2025-10-02 11:47:46,2025-10-02 11:48:55,-6.210000,5.120000,-3.450000,0.768815,TracersDispersionEvent_TS2_2025-10-02T11:47:46...,8.756769,135.167075,123.973199
6,42,2025-10-02 13:24:18,2025-10-02 13:25:22,-6.210000,5.120000,-3.450000,0.516902,TracersDispersionEvent_TS2_2025-10-02T13:24:18...,8.756769,135.167075,123.973199
7,61,2025-10-03 23:09:40,2025-10-03 23:10:53,-6.210000,5.120000,-3.450000,0.250695,TracersDispersionEvent_TS2_2025-10-03T23:09:40...,8.756769,135.167075,123.973199
8,64,2025-10-04 05:36:14,2025-10-04 05:37:22,-6.210000,5.120000,-3.450000,0.724066,TracersDispersionEvent_TS2_2025-10-04T05:36:14...,8.756769,135.167075,123.973199
9,65,2025-10-04 07:13:06,2025-10-04 07:14:36,-6.210000,5.120000,-3.450000,0.627554,TracersDispersionEvent_TS2_2025-10-04T07:13:06...,8.756769,135.167075,123.973199


In [15]:
labels = []

for _, row in df.iterrows():    
    cone_angle = row.cone_angle
    clock_angle = row.clock_angle 
    Bx = row.Bx
    B = row.B
    
    if np.abs(Bx) / B > 0.8:
        label = 'BigBx'
    elif clock_angle > 305 or clock_angle < 55:
        label = 'NorthwardIMF'
    elif (clock_angle > 55 and clock_angle < 155) or (clock_angle > 205 and clock_angle < 305):
        label = 'ByDominant'
    elif clock_angle > 155 and clock_angle < 205:
        label = 'SouthwardIMF'
    else:
        raise RuntimeError((cone_angle, clock_angle))

    labels.append(label)


In [16]:
df['label'] = labels
df = df.sort_values('start_time')

In [20]:
import glob

def get_aci_file(row):
    tstamp = row.start_time.strftime("%Y%m%d")
    return glob.glob(f'data/Sept30_Storm/aci/*{tstamp}*.cdf')[0]

def get_ead_file(row):
    tstamp = row.start_time.strftime("%Y%m%d")
    return glob.glob(f'data/Sept30_Storm/ead/*{tstamp}*.cdf')[0]
    
df['aci_file'] = df.apply(get_aci_file, axis=1)
df['ead_file'] = df.apply(get_ead_file, axis=1)
df

,index,start_time,end_time,Bx,By,Bz,score,file_name,B,cone_angle,clock_angle,label,aci_file,ead_file
0,0,2025-09-27 01:21:19,2025-09-27 01:22:37,-1.936180,-4.495037,0.769296,0.663395,TracersDispersionEvent_TS2_2025-09-27T01:21:19...,4.954389,113.004350,279.711705,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20250927_...,data/Sept30_Storm/ead/ts2_def_ead_20250927_v0....
1,5,2025-09-28 01:30:13,2025-09-28 01:31:32,-2.991730,4.040490,-4.695540,0.859231,TracersDispersionEvent_TS2_2025-09-28T01:30:13...,6.879252,115.778408,139.288158,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20250928_...,data/Sept30_Storm/ead/ts2_def_ead_20250928_v0....
2,20,2025-09-28 16:02:20,2025-09-28 16:03:41,-1.384180,-10.228882,-3.035928,0.573218,TracersDispersionEvent_TS2_2025-09-28T16:02:20...,10.759314,97.391556,253.469133,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20250928_...,data/Sept30_Storm/ead/ts2_def_ead_20250928_v0....
3,32,2025-09-30 11:28:56,2025-09-30 11:30:19,-8.338225,7.268442,-5.527518,0.688864,TracersDispersionEvent_TS2_2025-09-30T11:28:56...,12.365666,132.400167,127.252329,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20250930_...,data/Sept30_Storm/ead/ts2_def_ead_20250930_v0....
4,34,2025-10-01 16:29:31,2025-10-01 16:30:54,-6.210000,5.120000,-3.450000,0.664833,TracersDispersionEvent_TS2_2025-10-01T16:29:31...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251001_...,data/Sept30_Storm/ead/ts2_def_ead_20251001_v0....
5,41,2025-10-02 11:47:46,2025-10-02 11:48:55,-6.210000,5.120000,-3.450000,0.768815,TracersDispersionEvent_TS2_2025-10-02T11:47:46...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251002_...,data/Sept30_Storm/ead/ts2_def_ead_20251002_v0....
6,42,2025-10-02 13:24:18,2025-10-02 13:25:22,-6.210000,5.120000,-3.450000,0.516902,TracersDispersionEvent_TS2_2025-10-02T13:24:18...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251002_...,data/Sept30_Storm/ead/ts2_def_ead_20251002_v0....
7,61,2025-10-03 23:09:40,2025-10-03 23:10:53,-6.210000,5.120000,-3.450000,0.250695,TracersDispersionEvent_TS2_2025-10-03T23:09:40...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251003_...,data/Sept30_Storm/ead/ts2_def_ead_20251003_v0....
8,64,2025-10-04 05:36:14,2025-10-04 05:37:22,-6.210000,5.120000,-3.450000,0.724066,TracersDispersionEvent_TS2_2025-10-04T05:36:14...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251004_...,data/Sept30_Storm/ead/ts2_def_ead_20251004_v0....
9,65,2025-10-04 07:13:06,2025-10-04 07:14:36,-6.210000,5.120000,-3.450000,0.627554,TracersDispersionEvent_TS2_2025-10-04T07:13:06...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251004_...,data/Sept30_Storm/ead/ts2_def_ead_20251004_v0....


In [21]:
df.to_csv('output/Sept30_Storm_filtered.csv', index=0)
df

,index,start_time,end_time,Bx,By,Bz,score,file_name,B,cone_angle,clock_angle,label,aci_file,ead_file
0,0,2025-09-27 01:21:19,2025-09-27 01:22:37,-1.936180,-4.495037,0.769296,0.663395,TracersDispersionEvent_TS2_2025-09-27T01:21:19...,4.954389,113.004350,279.711705,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20250927_...,data/Sept30_Storm/ead/ts2_def_ead_20250927_v0....
1,5,2025-09-28 01:30:13,2025-09-28 01:31:32,-2.991730,4.040490,-4.695540,0.859231,TracersDispersionEvent_TS2_2025-09-28T01:30:13...,6.879252,115.778408,139.288158,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20250928_...,data/Sept30_Storm/ead/ts2_def_ead_20250928_v0....
2,20,2025-09-28 16:02:20,2025-09-28 16:03:41,-1.384180,-10.228882,-3.035928,0.573218,TracersDispersionEvent_TS2_2025-09-28T16:02:20...,10.759314,97.391556,253.469133,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20250928_...,data/Sept30_Storm/ead/ts2_def_ead_20250928_v0....
3,32,2025-09-30 11:28:56,2025-09-30 11:30:19,-8.338225,7.268442,-5.527518,0.688864,TracersDispersionEvent_TS2_2025-09-30T11:28:56...,12.365666,132.400167,127.252329,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20250930_...,data/Sept30_Storm/ead/ts2_def_ead_20250930_v0....
4,34,2025-10-01 16:29:31,2025-10-01 16:30:54,-6.210000,5.120000,-3.450000,0.664833,TracersDispersionEvent_TS2_2025-10-01T16:29:31...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251001_...,data/Sept30_Storm/ead/ts2_def_ead_20251001_v0....
5,41,2025-10-02 11:47:46,2025-10-02 11:48:55,-6.210000,5.120000,-3.450000,0.768815,TracersDispersionEvent_TS2_2025-10-02T11:47:46...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251002_...,data/Sept30_Storm/ead/ts2_def_ead_20251002_v0....
6,42,2025-10-02 13:24:18,2025-10-02 13:25:22,-6.210000,5.120000,-3.450000,0.516902,TracersDispersionEvent_TS2_2025-10-02T13:24:18...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251002_...,data/Sept30_Storm/ead/ts2_def_ead_20251002_v0....
7,61,2025-10-03 23:09:40,2025-10-03 23:10:53,-6.210000,5.120000,-3.450000,0.250695,TracersDispersionEvent_TS2_2025-10-03T23:09:40...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251003_...,data/Sept30_Storm/ead/ts2_def_ead_20251003_v0....
8,64,2025-10-04 05:36:14,2025-10-04 05:37:22,-6.210000,5.120000,-3.450000,0.724066,TracersDispersionEvent_TS2_2025-10-04T05:36:14...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251004_...,data/Sept30_Storm/ead/ts2_def_ead_20251004_v0....
9,65,2025-10-04 07:13:06,2025-10-04 07:14:36,-6.210000,5.120000,-3.450000,0.627554,TracersDispersionEvent_TS2_2025-10-04T07:13:06...,8.756769,135.167075,123.973199,ByDominant,data/Sept30_Storm/aci/ts2_l2_aci_ipd_20251004_...,data/Sept30_Storm/ead/ts2_def_ead_20251004_v0....
